## 0. Setup

Imports, paths and small helpers shared by every section (loading, displaying,
resolution checks). Images are kept in **RGB** order everywhere in this notebook
(OpenCV loads BGR, so `load_image` converts on load).

In [ ]:
from pathlib import Path
import os
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.dpi"] = 110

# Folder with the 10 images from Images.zip. Override with the P1_IMAGES env var.
def _find_images_dir() -> Path:
    if "P1_IMAGES" in os.environ:
        return Path(os.environ["P1_IMAGES"])
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / "data" / "raw").is_dir():
            return p / "data" / "raw"
    return Path("images")  # fallback: an `images/` folder next to the notebook

IMAGES_DIR = _find_images_dir()
MIN_SIDE = 400  # statement: results below 400x400 are not graded
print("Images:", IMAGES_DIR.resolve(), "| OpenCV", cv2.__version__)

In [ ]:
def load_image(n: int) -> np.ndarray:
    """Load image `n` (1..10) as uint8, RGB if colour, 2-D if greyscale."""
    matches = sorted(IMAGES_DIR.glob(f"{n}.*"))
    if not matches:
        raise FileNotFoundError(f"No image {n}.* in {IMAGES_DIR}")
    img = cv2.imread(str(matches[0]), cv2.IMREAD_UNCHANGED)
    if img.ndim == 3 and img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img


def describe(img: np.ndarray, name: str = "") -> None:
    """One-line summary: shape, dtype, channels, value range."""
    kind = "grey" if img.ndim == 2 else f"{img.shape[2]} channels"
    print(f"{name:>12} shape={img.shape} dtype={img.dtype} {kind} "
          f"min={img.min()} max={img.max()} mean={img.mean():.1f}")


def check_resolution(img: np.ndarray, name: str = "result") -> None:
    """Fail loudly if an image falls below the 400x400 minimum."""
    h, w = img.shape[:2]
    if h < MIN_SIDE or w < MIN_SIDE:
        raise ValueError(f"{name} is {w}x{h}: below the {MIN_SIDE}x{MIN_SIDE} minimum")


def resize_max_side(img: np.ndarray, max_side: int) -> np.ndarray:
    """Downscale so the longest side is `max_side` (never upscales, keeps aspect).
    Resizing is allowed by the statement; the result must stay >= 400x400."""
    h, w = img.shape[:2]
    scale = max_side / max(h, w)
    if scale >= 1:
        return img
    out = cv2.resize(img, (round(w * scale), round(h * scale)), interpolation=cv2.INTER_AREA)
    check_resolution(out, "resized image")
    return out

In [ ]:
def show(images: dict, cols: int = 2, size: float = 5.0, title: str | None = None) -> None:
    """Show {title: image} side by side. Greyscale gets a grey colormap."""
    n = len(images)
    cols = min(cols, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows), squeeze=False)
    for ax, (t, im) in zip(axes.flat, images.items()):
        ax.imshow(im, cmap="gray" if im.ndim == 2 else None, vmin=0, vmax=255 if im.dtype == np.uint8 else None)
        ax.set_title(t)
        ax.axis("off")
    for ax in list(axes.flat)[n:]:
        ax.axis("off")
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def show_hist(images: dict, bins: int = 256) -> None:
    """Per-channel histograms for each {title: image}."""
    fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 3), squeeze=False)
    for ax, (t, im) in zip(axes.flat, images.items()):
        if im.ndim == 2:
            ax.hist(im.ravel(), bins=bins, range=(0, 256), color="gray")
        else:
            for c, col in enumerate(("r", "g", "b")):
                ax.hist(im[..., c].ravel(), bins=bins, range=(0, 256), color=col, alpha=0.5)
        ax.set_title(t)
    plt.tight_layout()
    plt.show()


def check_generic(fn, **params) -> None:
    """Run `fn(img, **params)` on synthetic images of different sizes and channel
    counts (the statement penalises methods that only work on one image) and check
    the output keeps the input's shape. Adapt if your method changes the shape on purpose."""
    rng = np.random.default_rng(0)
    cases = {
        "grey 450x620": rng.integers(0, 256, (450, 620), dtype=np.uint8),
        "rgb 517x401": rng.integers(0, 256, (517, 401, 3), dtype=np.uint8),
        "rgb 401x900": rng.integers(0, 256, (401, 900, 3), dtype=np.uint8),
    }
    for label, img in cases.items():
        t0 = time.perf_counter()
        out = fn(img, **params)
        dt = time.perf_counter() - t0
        status = "ok" if out.shape == img.shape else f"SHAPE CHANGED -> {out.shape}"
        print(f"{label:>14}: {status}, dtype={out.dtype}, {dt:.2f}s")